# Clustering of Cryptocurrencies Based on Closing Prices

This notebook performs unsupervised clustering on daily closing price time series for a set of cryptocurrencies.  
The data come from the *Close* sheet of the provided `crypto.xlsx` file.  Because different cryptocurrencies started trading at different times, there are missing values at the beginning of some series.  Our approach follows common preprocessing steps suggested by the time‑series clustering literature:

- **Impute missing values** using interpolation.  GeeksforGeeks describes interpolation as a technique to estimate missing values based on surrounding data points rather than simply using the mean or median【906826468282410†L432-L469】.
- **Standardize** each time series before clustering.  Without standardization, variables with larger variation dominate the Euclidean distance metric used by k‑means.  Dmitrijs Kass’ article on k‑means clustering notes that clustering results depend on variable variation and recommends standardizing data prior to clustering【281075077385498†L143-L146】.
- **Use k‑means clustering** to group cryptocurrencies with similar price dynamics.  Time‑series clustering often uses transformations such as standardization and interpolation to remove noise before applying clustering algorithms【82817561445588†L110-L118】.

We will determine an appropriate number of clusters by computing the silhouette score for a range of `k` values and choose the `k` that maximizes this score.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns


## Load the closing price data

The raw closing‐price sheet contains two header rows.  The second header row (index 2) contains the tickers.  We read the sheet specifying `header=2`, rename the first column to `Date`, and convert it to `datetime`.  Next we convert each column to numeric and handle missing values with linear interpolation, forward fill and backward fill.  Columns containing no data or constant values are dropped.


In [ ]:
# Load the excel sheet with proper header
file_path = '/home/oai/share/crypto.xlsx'
# Use header=2 because the tickers are stored in the third row (0‑based indexing)
df = pd.read_excel(file_path, sheet_name='Close', header=2)
# Rename the first column to 'Date'
df = df.rename(columns={'Unnamed: 0': 'Date'})
# Convert to datetime
df['Date'] = pd.to_datetime(df['Date'])

# List of cryptocurrency columns (all columns except 'Date')
crypto_cols = df.columns[1:]

# Convert price columns to numeric and impute missing values
for col in crypto_cols:
    # convert to numeric, coerce errors to NaN
    df[col] = pd.to_numeric(df[col], errors='coerce')
    # interpolate missing values using linear interpolation
    df[col] = df[col].interpolate(method='linear', limit_direction='both')
    # forward fill then backward fill to handle leading/trailing NaNs
    df[col] = df[col].ffill().bfill()

# Drop columns that are entirely NaN or constant
cols_to_drop = []
for col in crypto_cols:
    # if all values are NaN or the column has only one unique value
    if df[col].isna().all() or df[col].nunique() <= 1:
        cols_to_drop.append(col)

df = df.drop(columns=cols_to_drop)
print(f"Dropped columns: {cols_to_drop}")

# Update crypto_cols after dropping
crypto_cols = df.columns[1:]

# Display basic information
print(f"Data shape after preprocessing: {df.shape}")
print(df.head())


## Prepare data for clustering

Each cryptocurrency’s time series is arranged in rows so that we can cluster them based on their temporal pattern.  We standardize each series individually (subtract its mean and divide by its standard deviation) to remove differences in scale.  After standardization, we compute the silhouette score for a range of cluster numbers to select an appropriate `k`.


In [ ]:
# Transpose the data: rows represent cryptocurrencies and columns represent time steps
X = df[crypto_cols].T
# Z‑score standardization across time for each cryptocurrency
X_standardized = (X - X.mean(axis=1).values.reshape(-1, 1)) / X.std(axis=1).values.reshape(-1, 1)
# Replace any NaNs resulting from zero variance with zeros
X_standardized = X_standardized.fillna(0)

# Determine the optimal number of clusters using silhouette score
ks = range(2, 10)
silhouette_scores = []
inertias = []
for k in ks:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_standardized)
    sil_score = silhouette_score(X_standardized, labels)
    silhouette_scores.append(sil_score)
    inertias.append(kmeans.inertia_)

# Plot silhouette scores and inertia to help decide on k
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(list(ks), silhouette_scores, marker='o')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Silhouette score')
plt.title('Silhouette score vs. k')

plt.subplot(1,2,2)
plt.plot(list(ks), inertias, marker='o')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Sum of squared distances (Inertia)')
plt.title('Elbow method (inertia)')
plt.tight_layout()
plt.show()

# Choose the k that maximizes the silhouette score
optimal_k = ks[np.argmax(silhouette_scores)]
print(f"Optimal number of clusters: {optimal_k}")

# Fit k‑means with the optimal k
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_labels = kmeans_final.fit_predict(X_standardized)

# Create a DataFrame to show cluster assignments
cluster_results = pd.DataFrame({'Crypto': X_standardized.index, 'Cluster': cluster_labels})
cluster_results = cluster_results.sort_values('Cluster').reset_index(drop=True)
cluster_results


## Visualize clusters in 2‑D space

To get an intuition for how the cryptocurrencies are grouped, we reduce the dimensionality of the standardized data using principal component analysis (PCA).  We then plot the first two principal components and color the points by cluster assignment.


In [ ]:
# Reduce to 2 dimensions for visualization using PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_standardized)

# Plot the PCA results with cluster labels
plt.figure(figsize=(8,6))
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=cluster_labels, palette='tab10', s=100)
plt.title('PCA of Standardized Crypto Time Series')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(title='Cluster')
plt.show()
